In [1]:
library(affy)
library(GEOquery)
library(tidyverse)

Loading required package: BiocGenerics

Loading required package: generics


Attaching package: 'generics'


The following objects are masked from 'package:base':

    as.difftime, as.factor, as.ordered, intersect, is.element, setdiff,
    setequal, union



Attaching package: 'BiocGenerics'


The following objects are masked from 'package:stats':

    IQR, mad, sd, var, xtabs


The following objects are masked from 'package:base':

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, is.unsorted, lapply, Map, mapply, match, mget,
    order, paste, pmax, pmax.int, pmin, pmin.int, Position, rank,
    rbind, Reduce, rownames, sapply, saveRDS, table, tapply, unique,
    unsplit, which.max, which.min


Loading required package: Biobase

Welcome to Bioconductor

    Vignettes contain introductory material; view with
    'browseVignettes()'. To cite Bioconductor, see
    'citation("Bioba

In [2]:
# defining datasets
training_set   <- "GSE42568"
validation_set <- c("GSE21653", "GSE20711", "GSE88770")

curr <- validation_set[2]

In [3]:
# reading in .cel files
raw_data <- ReadAffy(celfile.path = paste0("../data/cel_files/", curr))

# RMA normalize data
normalized_data <- rma(raw_data)

# get expression data
normalized_expr <- as.data.frame(exprs(normalized_data))
normalized_expr <- tibble::rownames_to_column(normalized_expr, var = "ID")

# Clean column names - extract just GSM IDs
colnames(normalized_expr) <- gsub("_.*", "", colnames(normalized_expr))

head(normalized_expr)


Warning message:
"replacing previous import 'AnnotationDbi::head' by 'utils::head' when loading 'hgu133plus2cdf'"
Warning message:
"replacing previous import 'AnnotationDbi::tail' by 'utils::tail' when loading 'hgu133plus2cdf'"




Background correcting
Normalizing
Calculating Expression


,ID,GSM519722.CEL.gz,GSM519723.CEL.gz,GSM519724.CEL.gz,GSM519725.CEL.gz,GSM519726.CEL.gz,GSM519727.CEL.gz,GSM519728.CEL.gz,GSM519729.CEL.gz,GSM519730.CEL.gz,⋯,GSM519803.CEL.gz,GSM519804.CEL.gz,GSM519805.CEL.gz,GSM519806.CEL.gz,GSM519807.CEL.gz,GSM519808.CEL.gz,GSM519809.CEL.gz,GSM519810.CEL.gz,GSM519811.CEL.gz,GSM519812.CEL.gz
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,1007_s_at,10.187052,9.652073,9.243115,9.665201,10.232769,10.278378,10.099109,10.287591,10.173119,⋯,9.485151,10.463529,9.595977,10.819352,9.782994,9.221509,10.057569,9.604759,10.017669,9.780783
2,1053_at,7.127039,6.863698,7.034919,6.911693,7.634268,7.007954,7.427445,8.161011,7.824778,⋯,7.234350,6.717380,7.132356,8.113843,6.127102,6.751229,6.432357,6.711111,6.705162,6.357043
3,117_at,6.887078,6.050344,6.631097,6.465884,6.318923,7.249174,5.825132,5.704061,6.100853,⋯,6.635512,6.356327,6.384561,5.461238,6.716387,6.643741,5.837846,6.927909,4.745112,7.269059
4,121_at,6.885332,7.185804,7.044363,6.599302,6.764104,7.151394,6.651394,7.008276,6.705128,⋯,6.671261,6.319033,6.335060,6.006954,6.253306,6.926910,7.095445,6.730393,6.428529,6.557090
5,1255_g_at,2.837355,2.647247,2.329934,2.563766,2.473097,2.483178,2.423432,2.306350,2.509102,⋯,2.864158,2.186710,2.513868,2.431490,2.514999,2.449755,2.529823,2.770067,2.807189,2.585510
6,1294_at,7.572196,6.627245,7.435874,7.353535,6.128658,7.072954,5.831950,6.968410,7.646905,⋯,7.301242,7.768421,7.509334,7.781010,7.015057,6.666444,6.719635,7.232949,7.803147,7.222269


In [4]:
# map probe IDs to gene symbols
gse <- getGEO(curr, GSEMatrix = TRUE)

Found 1 file(s)

GSE20711_series_matrix.txt.gz



In [5]:
# fetch feature data to get ID - gene symbol mapping
feature_data <- gse[[paste0(curr, "_series_matrix.txt.gz")]]@featureData@data

# subset
feature_data <- feature_data[c(1, 11)]

head(feature_data)

,ID,Gene Symbol
,<chr>,<chr>
1007_s_at,1007_s_at,DDR1 /// MIR4640
1053_at,1053_at,RFC2
117_at,117_at,HSPA6
121_at,121_at,PAX8
1255_g_at,1255_g_at,GUCA1A
1294_at,1294_at,MIR5193 /// UBA7


In [6]:
normalized_expr <- normalized_expr |>
  inner_join(feature_data, by = "ID")

head(normalized_expr)
dim(normalized_expr)

,ID,GSM519722.CEL.gz,GSM519723.CEL.gz,GSM519724.CEL.gz,GSM519725.CEL.gz,GSM519726.CEL.gz,GSM519727.CEL.gz,GSM519728.CEL.gz,GSM519729.CEL.gz,GSM519730.CEL.gz,⋯,GSM519804.CEL.gz,GSM519805.CEL.gz,GSM519806.CEL.gz,GSM519807.CEL.gz,GSM519808.CEL.gz,GSM519809.CEL.gz,GSM519810.CEL.gz,GSM519811.CEL.gz,GSM519812.CEL.gz,Gene Symbol
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
1,1007_s_at,10.187052,9.652073,9.243115,9.665201,10.232769,10.278378,10.099109,10.287591,10.173119,⋯,10.463529,9.595977,10.819352,9.782994,9.221509,10.057569,9.604759,10.017669,9.780783,DDR1 /// MIR4640
2,1053_at,7.127039,6.863698,7.034919,6.911693,7.634268,7.007954,7.427445,8.161011,7.824778,⋯,6.717380,7.132356,8.113843,6.127102,6.751229,6.432357,6.711111,6.705162,6.357043,RFC2
3,117_at,6.887078,6.050344,6.631097,6.465884,6.318923,7.249174,5.825132,5.704061,6.100853,⋯,6.356327,6.384561,5.461238,6.716387,6.643741,5.837846,6.927909,4.745112,7.269059,HSPA6
4,121_at,6.885332,7.185804,7.044363,6.599302,6.764104,7.151394,6.651394,7.008276,6.705128,⋯,6.319033,6.335060,6.006954,6.253306,6.926910,7.095445,6.730393,6.428529,6.557090,PAX8
5,1255_g_at,2.837355,2.647247,2.329934,2.563766,2.473097,2.483178,2.423432,2.306350,2.509102,⋯,2.186710,2.513868,2.431490,2.514999,2.449755,2.529823,2.770067,2.807189,2.585510,GUCA1A
6,1294_at,7.572196,6.627245,7.435874,7.353535,6.128658,7.072954,5.831950,6.968410,7.646905,⋯,7.768421,7.509334,7.781010,7.015057,6.666444,6.719635,7.232949,7.803147,7.222269,MIR5193 /// UBA7


[1] 54675    92

In [7]:
# For multiple probes → one gene: take average
# For one probe → multiple genes: delete

normalized_expr <- normalized_expr |>
  # Remove rows where gene symbol is empty or maps to multiple genes
  filter(!grepl("///", `Gene Symbol`)) |>
  filter(`Gene Symbol` != "") |>

  # Group by gene symbol and average across probes
  group_by(`Gene Symbol`) |>
  summarise(across(where(is.numeric), mean)) |>
  ungroup() |>
  rename(gene_symbol = `Gene Symbol`)

dim(normalized_expr)
head(normalized_expr)

[1] 21655    91

gene_symbol,GSM519722.CEL.gz,GSM519723.CEL.gz,GSM519724.CEL.gz,GSM519725.CEL.gz,GSM519726.CEL.gz,GSM519727.CEL.gz,GSM519728.CEL.gz,GSM519729.CEL.gz,GSM519730.CEL.gz,⋯,GSM519803.CEL.gz,GSM519804.CEL.gz,GSM519805.CEL.gz,GSM519806.CEL.gz,GSM519807.CEL.gz,GSM519808.CEL.gz,GSM519809.CEL.gz,GSM519810.CEL.gz,GSM519811.CEL.gz,GSM519812.CEL.gz
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
A1BG,6.119304,6.479491,6.017579,6.061706,5.581822,5.745512,4.886712,5.426486,5.901916,⋯,6.828642,6.224153,5.904253,7.912051,5.955679,5.698015,6.037793,5.715892,5.262424,5.574846
A1BG-AS1,4.600823,4.603668,5.055174,4.647037,4.182169,4.702075,4.326911,4.773466,4.570657,⋯,5.165543,5.413369,4.413916,5.295210,4.995111,4.816337,4.585186,4.774603,4.412655,4.546656
A1CF,3.121685,3.268132,3.115821,3.169886,3.329781,3.369047,3.217963,3.303424,3.170489,⋯,3.436043,3.180104,3.173097,3.547779,3.180838,3.681620,3.463427,3.442144,3.177729,3.593152
A2M,7.234631,7.518227,7.381913,7.575412,8.363513,7.212989,7.100545,7.251371,7.344661,⋯,6.895032,7.326967,7.521037,6.383923,7.135244,7.712773,7.493888,7.007678,7.672010,7.781730
A2M-AS1,4.909697,5.186144,7.623446,6.109816,6.142342,5.984419,4.648674,5.765407,5.374764,⋯,4.911827,7.175151,6.366014,4.906894,5.463223,4.654847,5.819628,5.777034,6.470250,6.035571
A2ML1,2.937211,2.967149,3.199144,2.913530,3.081503,5.443215,3.253625,5.462837,2.760528,⋯,3.035323,2.656849,2.834887,3.471181,2.745098,3.112381,3.015567,2.558184,3.231751,2.854460


In [12]:
# access the clinical metadata
clinical <- pData(gse[[1]])

colnames(clinical)
head(clinical)



[1] "title"                   "geo_accession"          
 [3] "status"                  "submission_date"        
 [5] "last_update_date"        "type"                   
 [7] "channel_count"           "source_name_ch1"        
 [9] "organism_ch1"            "characteristics_ch1"    
[11] "characteristics_ch1.1"   "characteristics_ch1.2"  
[13] "characteristics_ch1.3"   "characteristics_ch1.4"  
[15] "characteristics_ch1.5"   "characteristics_ch1.6"  
[17] "characteristics_ch1.7"   "characteristics_ch1.8"  
[19] "characteristics_ch1.9"   "characteristics_ch1.10" 
[21] "characteristics_ch1.11"  "characteristics_ch1.12" 
[23] "characteristics_ch1.13"  "molecule_ch1"           
[25] "extract_protocol_ch1"    "label_ch1"              
[27] "label_protocol_ch1"      "taxid_ch1"              
[29] "hyb_protocol"            "scan_protocol"          
[31] "description"             "data_processing"        
[33] "platform_id"             "contact_name"           
[35] "contact_email"           "contact_phone"          
[37] "contact_laboratory"      "contact_department"     
[39] "contact_institute"       "contact_address"        
[41] "contact_city"            "contact_state"          
[43] "contact_zip/postal_code" "contact_country"        
[45] "supplementary_file"      "data_row_count"         
[47] "age (bin):ch1"           "e.os:ch1"               
[49] "e.rfs:ch1"               "er status:ch1"          
[51] "grade:ch1"               "her2 status:ch1"        
[53] "methylation barcode:ch1" "node:ch1"               
[55] "quality control:ch1"     "size (bin):ch1"         
[57] "subtypege:ch1"           "subtypeihc:ch1"         
[59] "t.os:ch1"                "t.rfs:ch1"

,title,geo_accession,status,submission_date,last_update_date,type,channel_count,source_name_ch1,organism_ch1,characteristics_ch1,⋯,grade:ch1,her2 status:ch1,methylation barcode:ch1,node:ch1,quality control:ch1,size (bin):ch1,subtypege:ch1,subtypeihc:ch1,t.os:ch1,t.rfs:ch1
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
GSM519722,Breast tumor from patient P_1 (expression data),GSM519722,Public on Oct 19 2011,Mar 09 2010,Jun 17 2021,RNA,1,Breast tumor,Homo sapiens,quality control: 1,⋯,3,1,4690141010_D,1,1,1,HER2+,HER2,8.28 y,0.93
GSM519723,Breast tumor from patient P_2 (expression data),GSM519723,Public on Oct 19 2011,Mar 09 2010,Jun 17 2021,RNA,1,Breast tumor,Homo sapiens,quality control: 1,⋯,3,1,4690141010_J,1,1,1,HER2+,HER2,3.39 y,1.22
GSM519724,Breast tumor from patient P_3 (expression data),GSM519724,Public on Oct 19 2011,Mar 09 2010,Jun 17 2021,RNA,1,Breast tumor,Homo sapiens,quality control: 1,⋯,3,1,4690141011_A,0,1,1,HER2+,HER2,8.64 y,8.64
GSM519725,Breast tumor from patient P_4 (expression data),GSM519725,Public on Oct 19 2011,Mar 09 2010,Jun 17 2021,RNA,1,Breast tumor,Homo sapiens,quality control: 1,⋯,3,0,4690141010_I,1,1,1,LumB,LumB,8.4 y,8.4
GSM519726,Breast tumor from patient P_6 (expression data),GSM519726,Public on Oct 19 2011,Mar 09 2010,Jun 17 2021,RNA,1,Breast tumor,Homo sapiens,quality control: 1,⋯,3,1,4690141011_E,0,1,0,Basal,HER2,7.21 y,7.21
GSM519727,Breast tumor from patient P_7 (expression data),GSM519727,Public on Oct 19 2011,Mar 09 2010,Jun 17 2021,RNA,1,Breast tumor,Homo sapiens,quality control: 1,⋯,3,0,4690141010_K,1,1,0,Basal,Basal,8.19 y,7.01


In [14]:
t(head(clinical, 1))

,GSM519722
title,Breast tumor from patient P_1 (expression data)
geo_accession,GSM519722
status,Public on Oct 19 2011
submission_date,Mar 09 2010
last_update_date,Jun 17 2021
type,RNA
channel_count,1
source_name_ch1,Breast tumor
organism_ch1,Homo sapiens
characteristics_ch1,quality control: 1


In [15]:
clinical_filtered <- clinical %>%
  mutate(relapse_free_time = as.integer(as.numeric(`t.rfs:ch1`) * 365.25)) %>%
  mutate(relapse_free_event = as.integer(as.numeric(gsub("e.rfs: ", "", `characteristics_ch1.10`))))

Warning message:
"There was 1 warning in `mutate()`.
ℹ In argument: `relapse_free_time = as.integer(as.numeric(`t.rfs:ch1`) *
  365.25)`.
Caused by warning:
! NAs introduced by coercion"
Warning message:
"There was 1 warning in `mutate()`.
ℹ In argument: `relapse_free_event = as.integer(as.numeric(gsub("e.rfs: ", "",
  characteristics_ch1.10)))`.
Caused by warning:
! NAs introduced by coercion"


In [ ]:
# Extract relevant columns
clinical_filtered <- clinical_filtered[, c(
  "geo_accession",
  "tumor:ch1",
  "relapse_free_event",
  "relapse_free_time"
)]

# Rename columns
colnames(clinical_filtered) <- c(
  "sample_id",
  "is_tumor",
  "relapse_free_event",
  "relapse_free_time"
)

# encode is_tumor as binary
clinical$is_tumor <- ifelse(clinical$is_tumor == "breast cancer tumor", 1, 0)

cols_to_convert <- c("is_tumor", "relapse_free_event", "relapse_free_time")

In [17]:
head(clinical_filtered)

,sample_id,relapse_free_event,relapse_free_time
,<chr>,<int>,<int>
GSM519722,GSM519722,1,339
GSM519723,GSM519723,1,445
GSM519724,GSM519724,0,3155
GSM519725,GSM519725,0,3068
GSM519726,GSM519726,0,2633
GSM519727,GSM519727,1,2560


In [19]:
# save datasets in datasets folder for persistent storage
write.csv(normalized_expr, "../datasets/csv_files/expression_matrix_test_two.csv")
write.csv(clinical_filtered, "../datasets/csv_files/clinical_metadata_test_two.csv")